# GPN-MSA: basic example

## Setup

In [1]:
import numpy as np
import torch
from gpn.data import GenomeMSA, Tokenizer
from transformers import AutoModelForPreTraining

In [2]:
model_path = "../MSA-GPT-Checkpoint"  # path to the model checkpoint
# see README in https://huggingface.co/datasets/songlab/multiz100way for faster queries

# download the MSA dataset from https://huggingface.co/datasets/songlab/multiz100way/resolve/main/89.zarr.zip and provide the path
msa_path = "../analysis/89.zarr"

## Loading data

Example region: chr6:31575665-31575793

[UCSC Genome Browser view](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr6%3A31575665%2D31575793&hgsid=1726885238_vIMnX2NGEluaKCXVZjeTkj97aydM) 

In [3]:
genome_msa = GenomeMSA(msa_path, in_memory=False)  # can take a minute or two

Loading MSA...
Loading MSA... Done


/home/david/.conda/envs/msa_gpn/lib/python3.11/site-packages/zarr/core/group.py:3301: UserWarning: Object at .snakemake_timestamp is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [4]:
# untokenized msa
raw_msa = genome_msa.get_msa("6", 31575665, 31575793, strand="+", tokenize=False)
print(raw_msa.shape)
raw_msa

(128, 90)


array([[b'A', b'A', b'-', ..., b'-', b'-', b'-'],
       [b'C', b'G', b'-', ..., b'-', b'-', b'-'],
       [b'A', b'A', b'A', ..., b'-', b'-', b'-'],
       ...,
       [b'T', b'T', b'T', ..., b'-', b'-', b'-'],
       [b'C', b'T', b'C', ..., b'-', b'-', b'-'],
       [b'C', b'C', b'C', ..., b'-', b'-', b'-']],
      shape=(128, 90), dtype='|S1')

In [5]:
# tokenized msa
msa = genome_msa.get_msa("6", 31575665, 31575793, strand="+", tokenize=True)
msa

array([[1, 1, 0, ..., 0, 0, 0],
       [2, 3, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [4, 4, 4, ..., 0, 0, 0],
       [2, 4, 2, ..., 0, 0, 0],
       [2, 2, 2, ..., 0, 0, 0]], shape=(128, 90), dtype=uint8)

In [6]:
msa = torch.tensor(np.expand_dims(msa, 0).astype(np.int64))
msa

tensor([[[1, 1, 0,  ..., 0, 0, 0],
         [2, 3, 0,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [4, 4, 4,  ..., 0, 0, 0],
         [2, 4, 2,  ..., 0, 0, 0],
         [2, 2, 2,  ..., 0, 0, 0]]])

In [7]:
# separating human from rest of species
input_ids, aux_features = msa[:, :, 0], msa[:, :, 1:]
input_ids.shape, aux_features.shape

(torch.Size([1, 128]), torch.Size([1, 128, 89]))

## Masked language modeling

In [8]:
# model = AutoModel.from_pretrained(model_path)
model = AutoModelForPreTraining.from_pretrained(model_path)
model.eval();

In [9]:
raw_msa[76:79, 0]  # Start codon

array([b'A', b'T', b'G'], dtype='|S1')

In [10]:
tokenizer = Tokenizer()
pos = 76  # Let's mask the A and check the model predictions
input_ids[0, pos] = tokenizer.mask_token_id()

In [11]:
with torch.no_grad():
    output = model(input_ids=input_ids, aux_features=aux_features)
print(f"Output shape: {output.logits.shape}")

In [12]:
predicted_token_ids = torch.argmax(output.logits, dim=-1)

In [13]:
predicted_token_ids

tensor([[1, 1, 3, 1, 1, 1, 3, 4, 4, 1, 3, 1, 1, 1, 3, 2, 3, 3, 4, 4, 3, 2, 3, 4,
         2, 1, 3, 2, 3, 2, 3, 4, 3, 4, 1, 1, 4, 4, 1, 1, 4, 4, 3, 1, 4, 3, 2, 3,
         4, 4, 3, 4, 2, 1, 3, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 3, 3, 1, 2, 1, 1,
         1, 3, 1, 1, 1, 3, 3, 3, 2, 2, 3, 2, 3, 3, 3, 3, 1, 2, 1, 1, 2, 3, 1, 3,
         1, 1, 3, 3, 1, 3, 1, 1, 3, 3, 1, 3, 3, 1, 3, 3, 1, 2, 1, 1, 3, 3, 1, 3,
         3, 1, 2, 3, 4, 3, 1, 1]])

In [14]:
nucleotides = list("ACGT")

to_nucleotide_dict = {tokenizer.vocab.index(nc): nc for nc in nucleotides}
to_nucleotide_dict

{1: 'A', 2: 'C', 3: 'G', 4: 'T'}

In [15]:
for i in predicted_token_ids.squeeze():
    if i.item() in to_nucleotide_dict:
        print(to_nucleotide_dict[i.item()])
    else:
        print("Unknown token:", i.item())

A
A
G
A
A
A
G
T
T
A
G
A
A
A
G
C
G
G
T
T
G
C
G
T
C
A
G
C
G
C
G
T
G
T
A
A
T
T
A
A
T
T
G
A
T
G
C
G
T
T
G
T
C
A
G
C
C
C
C
C
C
C
A
C
C
C
G
G
A
C
A
A
A
G
A
A
A
G
G
G
C
C
G
C
G
G
G
G
A
C
A
A
C
G
A
G
A
A
G
G
A
G
A
A
G
G
A
G
G
A
G
G
A
C
A
A
G
G
A
G
G
A
C
G
T
G
A
A
